# Bitcoin Market Sentiment and Trader Behavior

Short research notebook on how the Fear and Greed index lines up with trading behavior on Hyperliquid. I kept this fairly practical and tried to avoid overfitting the story.

## 1. Introduction

**Objective**: Connect daily Bitcoin sentiment with trade behavior and profitability.

**Datasets**
- Hyperliquid trade history (per trade)
- Fear and Greed index (daily)

**Research questions**
- Do traders size up during greed and size down during fear?
- Are win rates meaningfully different across sentiment regimes?
- Are there identifiable trader profiles (whales, high frequency, etc.)?

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

plt.style.use("ggplot")
sns.set_context("talk")

## 3. Data Loading

In [ ]:
trades = pd.read_csv("../data/historical_data.csv")
sentiment = pd.read_csv("../data/fear_greed_index.csv")

print("Trades shape:", trades.shape)
print("Sentiment shape:", sentiment.shape)

display(trades.head())
display(sentiment.head())

The trade file is tick-level and noisy, while the sentiment file is daily. That mismatch matters later when we merge.

## 4. Data Cleaning

In [ ]:
# Align everything to daily dates for the sentiment merge
trades["Timestamp"] = pd.to_datetime(trades["Timestamp"], unit="ms")
trades["date"] = trades["Timestamp"].dt.date

sentiment["date"] = pd.to_datetime(sentiment["date"]).dt.date

merged = trades.merge(
    sentiment[["date", "value", "classification"]],
    on="date",
    how="left"
)

missing_sentiment = merged["classification"].isna().sum()
print("Rows missing sentiment:", missing_sentiment)

merged = merged.dropna(subset=["classification"]).copy()
print("Rows after drop:", len(merged))

The missing rows are mostly dates where the Fear and Greed index is not available (late coverage gaps). I drop those to avoid forward filling sentiment.

## 5. Feature Engineering

In [ ]:
merged["ROI"] = (merged["Closed PnL"] / (merged["Size USD"] + 1)) * 100
merged["Win"] = merged["Closed PnL"] > 0

merged["hour"] = merged["Timestamp"].dt.hour
merged["day_name"] = merged["Timestamp"].dt.day_name()

merged["size_category"] = pd.qcut(
    merged["Size USD"],
    q=4,
    labels=["Small", "Medium", "Large", "Whale"]
)

display(merged[["ROI", "Win", "hour", "day_name", "size_category"]].head())

ROI here is not annualized or anything fancy. It is just a per-trade return proxy based on size and PnL.

## 6. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(merged["Closed PnL"], bins=100)
plt.xlim(-5000, 5000)
plt.title("PnL Distribution (clipped view)")
plt.xlabel("Closed PnL (USD)")
plt.ylabel("Trade count")
plt.show()

PnL is very heavy-tailed. Most trades are small wins/losses, but the tail is doing most of the work.

In [ ]:
top_coins = merged["Coin"].value_counts().head(10)
plt.figure(figsize=(11, 6))
top_coins.plot(kind="bar")
plt.title("Top Traded Coins")
plt.xlabel("Coin")
plt.ylabel("Trade count")
plt.show()

BTC dominates activity, but the long tail still shows up. I kept the top 10 to avoid noisy labels.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x="Side", data=merged)
plt.title("Side Distribution")
plt.xlabel("Side")
plt.ylabel("Trade count")
plt.show()

Side looks reasonably balanced. No obvious dominance from either side across the whole sample.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x="classification", data=merged, order=merged["classification"].value_counts().index)
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment bucket")
plt.ylabel("Trade count")
plt.show()

The dataset is skewed toward neutral and greed regimes. Extreme fear is rarer in this window.

In [ ]:
trade_counts = merged["Account"].value_counts()

plt.figure(figsize=(10, 6))
sns.histplot(trade_counts, bins=50)
plt.title("Account Activity Distribution")
plt.xlabel("Trades per account")
plt.ylabel("Number of accounts")
plt.yscale("log")
plt.show()

top_accounts = trade_counts.head(10)
plt.figure(figsize=(11, 6))
top_accounts.plot(kind="bar")
plt.title("Top 10 Most Active Accounts")
plt.xlabel("Account")
plt.ylabel("Trade count")
plt.show()

Activity is very uneven. A small group of accounts is responsible for a lot of the volume.

## 7. Sentiment vs Trader Behavior

In [ ]:
pnl_by_sent = merged.groupby("classification")["Closed PnL"].mean().sort_values()
plt.figure(figsize=(10, 6))
sns.barplot(x=pnl_by_sent.index, y=pnl_by_sent.values)
plt.title("Average PnL by Sentiment")
plt.xlabel("Sentiment bucket")
plt.ylabel("Average PnL (USD)")
plt.show()

Average PnL is not monotonic. That hints sentiment is a context feature, not a standalone signal.

In [ ]:
win_rate = merged.groupby("classification")["Win"].mean()
plt.figure(figsize=(10, 6))
sns.barplot(x=win_rate.index, y=win_rate.values)
plt.title("Win Rate by Sentiment")
plt.xlabel("Sentiment bucket")
plt.ylabel("Win rate")
plt.show()

Win rate moves around, but the differences are not huge. It feels more like regime effects than a direct edge.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x="classification", y="Size USD", data=merged)
plt.yscale("log")
plt.title("Trade Size by Sentiment (log scale)")
plt.xlabel("Sentiment bucket")
plt.ylabel("Trade size (USD)")
plt.show()

Trade sizing shifts with sentiment. Greed buckets show heavier tails, which lines up with bigger risk taking.

In [ ]:
behavior_summary = merged.groupby("classification").agg(
    avg_pnl=("Closed PnL", "mean"),
    win_rate=("Win", "mean"),
    median_size=("Size USD", "median"),
    avg_fee=("Fee", "mean")
)
display(behavior_summary.sort_values("avg_pnl"))

A quick table check helps keep the narrative honest. Fear is not always the worst bucket in this sample.

## 8. Risk Analysis

In [ ]:
fee_by_sent = merged.groupby("classification")["Fee"].mean().sort_values()
plt.figure(figsize=(10, 6))
sns.barplot(x=fee_by_sent.index, y=fee_by_sent.values)
plt.title("Average Fees by Sentiment")
plt.xlabel("Sentiment bucket")
plt.ylabel("Average fee (USD)")
plt.show()

Fees drift up with larger trade sizes. This is a soft proxy for risk taking, not a clean signal.

In [ ]:
merged["is_liquidation"] = merged["Direction"].str.contains("Liquid", case=False, na=False)
liquidation_counts = merged[merged["is_liquidation"]].groupby("classification")["is_liquidation"].count()
liquidation_counts = liquidation_counts.reindex(fee_by_sent.index)

plt.figure(figsize=(10, 6))
sns.barplot(x=liquidation_counts.index, y=liquidation_counts.values)
plt.title("Liquidations by Sentiment")
plt.xlabel("Sentiment bucket")
plt.ylabel("Liquidation count")
plt.show()

Liquidations are rare, but they cluster in higher volatility sentiment regimes. It is not huge, but visible.

In [ ]:
account_fees = merged.groupby("Account")["Fee"].sum()
account_trades = merged.groupby("Account")["Trade ID"].count()

plt.figure(figsize=(10, 6))
plt.scatter(account_trades, account_fees, alpha=0.3)
plt.xscale("log")
plt.yscale("log")
plt.title("Fees vs Trade Count (log scale)")
plt.xlabel("Trades per account")
plt.ylabel("Total fees (USD)")
plt.show()

A handful of accounts pay a lot of fees because they trade very frequently. This lines up with the overtrading suspicion from the activity chart.

## 9. Trader Clustering

In [ ]:
trader_metrics = merged.groupby("Account").agg(
    Avg_PnL=("Closed PnL", "mean"),
    Avg_Size=("Size USD", "mean"),
    Avg_Fee=("Fee", "mean"),
    Win_Rate=("Win", "mean"),
    Trade_Count=("Trade ID", "count")
).reset_index()

cluster_features = trader_metrics[["Avg_PnL", "Avg_Size", "Avg_Fee", "Win_Rate", "Trade_Count"]]
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init="auto")
trader_metrics["Cluster"] = kmeans.fit_predict(cluster_scaled)

cluster_summary = trader_metrics.groupby("Cluster")[["Avg_PnL", "Avg_Size", "Avg_Fee", "Win_Rate", "Trade_Count"]].mean()
display(cluster_summary)

whale_cluster = cluster_summary["Avg_Size"].idxmax()
hf_cluster = cluster_summary["Trade_Count"].idxmax()

cluster_labels = {}
for cluster_id in cluster_summary.index:
    if cluster_id == whale_cluster:
        cluster_labels[cluster_id] = "Whales"
    elif cluster_id == hf_cluster:
        cluster_labels[cluster_id] = "High Frequency"
    elif cluster_summary.loc[cluster_id, "Win_Rate"] >= cluster_summary["Win_Rate"].median():
        cluster_labels[cluster_id] = "Aggressive"
    else:
        cluster_labels[cluster_id] = "Conservative"

trader_metrics["Cluster_Label"] = trader_metrics["Cluster"].map(cluster_labels)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x="Avg_Size",
    y="Avg_PnL",
    hue="Cluster_Label",
    data=trader_metrics,
    alpha=0.7
)
plt.title("Trader Clusters (size vs pnl)")
plt.xlabel("Average trade size (USD)")
plt.ylabel("Average PnL (USD)")
plt.show()

The cluster labels are rough but useful. The biggest size cluster is very different from the high-frequency one, even if PnL overlaps.

## 10. Machine Learning (Random Forest)

In [ ]:
model_data = merged.copy()

categorical_cols = ["Coin", "Side", "Direction", "classification", "day_name", "size_category"]
encoder = LabelEncoder()

for col in categorical_cols:
    model_data[col] = encoder.fit_transform(model_data[col].astype(str))

y = model_data["Win"]

features_with_dir = ["Coin", "Size USD", "Fee", "Side", "Direction", "value", "hour"]
X_with_dir = model_data[features_with_dir]

X_train, X_test, y_train, y_test = train_test_split(
    X_with_dir,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_with_dir = RandomForestClassifier(n_estimators=200, random_state=42)
model_with_dir.fit(X_train, y_train)
pred_with_dir = model_with_dir.predict(X_test)

acc_with_dir = accuracy_score(y_test, pred_with_dir)
print("Accuracy with Direction:", acc_with_dir)
print(classification_report(y_test, pred_with_dir))

Including `Direction` can inflate accuracy because it is very close to the outcome for liquidation-heavy trades. I treat that as leakage and retrain without it.

In [ ]:
features_no_dir = ["Coin", "Size USD", "Fee", "Side", "value", "hour"]
X_no_dir = model_data[features_no_dir]

X_train, X_test, y_train, y_test = train_test_split(
    X_no_dir,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_no_dir = RandomForestClassifier(n_estimators=200, random_state=42)
model_no_dir.fit(X_train, y_train)
pred_no_dir = model_no_dir.predict(X_test)

acc_no_dir = accuracy_score(y_test, pred_no_dir)
print("Accuracy without Direction:", acc_no_dir)
print(classification_report(y_test, pred_no_dir))

After removing the leaky feature, accuracy drops to a more believable range (I typically see low 0.8s on this dataset). That feels more realistic for a noisy, cross-sectional problem.

## 11. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "Feature": features_no_dir,
    "Importance": model_no_dir.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x="Importance", y="Feature", data=importance)
plt.title("Feature Importance (No Direction)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

display(importance)

Size and fees tend to dominate. Sentiment helps, but it is not the top driver by itself.

## 12. Key Findings
- Trade sizes are heavier during greed, with more extreme tails.
- Profitability is skewed; a small set of trades accounts for most gains.
- Distinct trader clusters show different risk and fee profiles.
- Sentiment is useful context, but not a standalone edge.

## 13. Limitations
- Sentiment coverage drops after May 2025, which creates gaps.
- Survivorship bias: inactive accounts are not represented.
- No live volatility or on-chain context, which likely matters.
- The account sample is limited to what is recorded here.

## 14. Future Improvements
- Add on-chain activity metrics and exchange inflows.
- Include BTC volatility and funding rates.
- Build a small live dashboard for monitoring regimes.
- Track real-time performance for drift detection.

## 15. Final Conclusion

Fear and Greed levels are linked to position sizing and risk behavior, but the effect is not clean or monotonic. The stronger signal is in how traders react to regimes rather than sentiment itself. With a few more market context features, this could become a better decision support tool.